# Open-Vocabulary Detection with LibreYOLOWorld (MVP)

Text-prompted YOLO: give it a list of class names as strings and it detects those classes. This branch (`agentic/yolo-world`) ships an **architecture + API scaffold** — the forward pass works end-to-end, CLIP text embeddings flow through the model, prompts are hot-swappable at inference time. Weights are random-init in this MVP; porting from [Tencent/YOLO-World](https://huggingface.co/Tencent/YOLO-World) is future work.

This notebook walks through:
1. CLIP text encoding for arbitrary prompts.
2. Forward pass on a synthetic image.
3. Hot-swapping prompts at runtime.
4. The user-facing `LibreYOLOWorld(...)(image)` API.

## 1. Install

In [ ]:
# pip install -e 'git+https://github.com/aalvsz/libreyolo@agentic/yolo-world#egg=libreyolo[yoloworld]'
import numpy as np
import torch
from PIL import Image

from libreyolo.models.yoloworld import LibreYOLOWorld, LibreYOLOWorldModel

## 2. Raw architecture — text encoder + vision encoder

CLIP produces L2-normalized text embeddings; our vision encoder outputs a spatial feature map projected into the same embedding space. Per-location class logits are cosine similarities scaled by a learnable temperature (CLIP convention).

In [ ]:
model = LibreYOLOWorldModel(imgsz=256)
embeds = model.text_encoder.encode([
    'a photo of a person',
    'a photo of a dog',
    'a traffic cone',
    'an umbrella',
])
print(f'text embeddings: {embeds.shape} (L2-normalized, dim={embeds.shape[-1]})')
print(f'pairwise similarity:\n{(embeds @ embeds.T).round(decimals=3)}')

## 3. Forward pass

Set prompts once, run as many images as you want through the same encoded-text cache.

In [ ]:
model.set_prompts(['person', 'bicycle', 'car', 'dog'])
x = torch.randn(1, 3, 256, 256)
out = model(x)
for k, v in out.items():
    if hasattr(v, 'shape'):
        print(f'  {k:<6}: {tuple(v.shape)}')
    else:
        print(f'  {k:<6}: {v}')

## 4. Hot-swap prompts (no model rebuild)

Changing `model.set_prompts(...)` just re-encodes the text; the vision backbone is unchanged. This is the key property of open-vocab detection — at deployment you don't retrain to add a new class, you just add the string.

In [ ]:
# Prompt list A
model.set_prompts(['cat', 'dog'])
print(f'2 prompts  → cls shape: {model(x)["cls"].shape}')
# Prompt list B — add custom categories on the fly
model.set_prompts(['red apple', 'green apple', 'banana', 'orange', 'grape'])
print(f'5 prompts  → cls shape: {model(x)["cls"].shape}')

## 5. User-facing API

The `LibreYOLOWorld` wrapper adapts the model into LibreYOLO's standard `Results` interface.

In [ ]:
# Save a synthetic test image
rng = np.random.default_rng(0)
img = Image.fromarray(rng.integers(0, 255, size=(400, 600, 3), dtype=np.uint8))
img.save('demo_input.jpg')

# Open-vocab detection
det = LibreYOLOWorld(prompts=['person', 'dog', 'traffic cone', 'bird'],
                     imgsz=256, device='cpu')
result = det('demo_input.jpg', conf=0.0, max_det=5)
res = result[0] if isinstance(result, list) else result
print(f'detections: {len(res.boxes)}')
if len(res.boxes) > 0:
    cls_ids = res.boxes.cls.long().tolist()
    conf = res.boxes.conf.tolist()
    for i, (c, s) in enumerate(zip(cls_ids, conf)):
        print(f'  {i:>2}. {det.prompts[c]!r}  conf={s:.3f}')

**IMPORTANT — MVP scope:** weights are random-init. Detections above are noise. The architecture, API, and plumbing are correct; accurate open-vocab detection requires porting [Tencent/YOLO-World](https://huggingface.co/Tencent/YOLO-World) weights, which is future work tracked in the blog post.